# # 04 — Cross-Subject LOSO Benchmark

This notebook evaluates cross-subject generalization using a **leave-one-subject-out (LOSO)** design.

## Goal

Measure how well each model generalizes to a completely unseen subject.

## Models compared

- **LDA**
- **Logistic Regression**
- **EEGNet**

## Evaluation design

For each held-out subject:

1. train on all remaining subjects
2. keep the held-out subject completely untouched until test time
3. for EEGNet, create validation data **only from the training subjects**
4. evaluate on the unseen test subject

## Why this notebook matters

This notebook is **not** the primary model-selection benchmark.
The primary benchmark remains the grouped within-subject benchmark.

However, LOSO is important because it shows:

- how much subject-specific dependence exists
- how much performance drops under real cross-subject generalization
- whether the final candidate model still behaves reasonably on unseen subjects

## Outputs

This notebook will produce:

- one LOSO result per `(held-out subject, model, seed)`
- subject-level summaries across seeds
- model-level LOSO summaries
- comparison against the within-subject benchmark

All outputs are saved under:

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/`

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family
from oddball_eeg.features import make_classical_features
from oddball_eeg.io import save_dataframe, save_experiment_metadata
from oddball_eeg.loader import load_subjects_grouped, make_subject_summary_table
from oddball_eeg.metrics import classification_metric_dict
from oddball_eeg.models import get_model
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.reproducibility import get_default_device
from oddball_eeg.summaries import (
    summarize_fold_results,
    summarize_model_results_across_subjects,
    summarize_subject_results_across_seeds,
)
from oddball_eeg.training import train_eegnet_one_fold
from oddball_eeg.visualization import (
    make_main_vs_control_table,
    make_model_ranking_table,
    make_subject_metric_table,
    plot_learning_curve,
    plot_main_vs_control,
    plot_model_ranking_bars,
    plot_subject_metric_bars,
    plot_subject_seed_stability,
    print_model_ranking,
    print_subject_model_summary,
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

In [ ]:
set_pandas_display_options()

## 1. Load configurations

We use:

- the frozen LOSO configuration for this notebook
- the main within-subject benchmark outputs for later comparison

In [ ]:
# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds003061_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds005863_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds006466_within_subject_max30_wide_timebin"
EXPERIMENT_NAME = "ds006480_within_subject_max30_wide_timebin"

experiment_family = get_experiment_family(EXPERIMENT_NAME)

cfg_main = experiment_family["main"]
cfg_perm = experiment_family["permuted"]
cfg_prestim = experiment_family["prestim"]
cfg_loso = experiment_family["loso"]

device = get_default_device()

save_experiment_metadata(
    cfg_loso,
    extra={
        "notebook": "04_cross_subject_loso.ipynb",
        "purpose": "Cross-subject leave-one-subject-out benchmark.",
        "main_experiment_key": EXPERIMENT_NAME,
    },
)

print("Experiment key :", EXPERIMENT_NAME)
print("LOSO experiment:", cfg_loso.name)
print("Dataset        :", cfg_loso.dataset.dataset)
print("Task           :", cfg_loso.dataset.task)
print("Subjects       :", cfg_loso.subjects.subjects)
print("Seeds          :", cfg_loso.seeds.seeds)
print("Signal config  :", cfg_loso.signal)
print("Crop window    :", cfg_loso.effective_crop_window())
print("Device         :", device)

print("\nLOSO result directory:")
print(cfg_loso.result_dir())

print("\nMain within-subject result directory:")
print(cfg_main.result_dir())

## 2. Load grouped chronological subject data and apply the main crop window

LOSO should use the same main signal window as the within-subject benchmark, so that the comparison stays meaningful.

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg_loso)

print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg_loso.dataset.dataset,
    task_name=cfg_loso.dataset.task,
    subjects=cfg_loso.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg_loso.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

In [ ]:
crop_window = cfg_loso.effective_crop_window()
assert crop_window is not None, "Expected a crop window for the LOSO benchmark."

data_loso = apply_crop_to_loaded_dataset(
    data_full,
    tmin=crop_window[0],
    tmax=crop_window[1],
)

subject_summary_loso = make_subject_summary_table(data_loso)
subject_summary_loso = round_numeric_columns(subject_summary_loso, decimals=4)

display(subject_summary_loso)

save_dataframe(
    subject_summary_loso,
    cfg_loso.table_dir("summaries") / "loaded_subject_summary_loso_cropped.csv",
)

## 3. Define LOSO helper functions

The core utilities already cover the within-subject benchmark.
For LOSO, we define a few notebook-local helpers to keep the minimal 6-file project structure simple.

In [ ]:
def concatenate_training_subjects_with_unique_groups(data, train_subjects):
    """
    Concatenate multiple training subjects into one training set.

    Groups are offset so that group IDs remain unique across subjects.
    """
    X_list = []
    y_list = []
    g_list = []

    group_offset = 0

    for sub in train_subjects:
        X_sub = data.X_by_sub[sub]
        y_sub = data.y_by_sub[sub]
        g_sub = data.groups_by_sub[sub]

        g_unique = g_sub + group_offset

        X_list.append(X_sub)
        y_list.append(y_sub)
        g_list.append(g_unique)

        group_offset = int(g_unique.max()) + 1

    X_train = np.concatenate(X_list, axis=0).astype(np.float32)
    y_train = np.concatenate(y_list, axis=0).astype(np.int64)
    g_train = np.concatenate(g_list, axis=0).astype(np.int64)

    return X_train, y_train, g_train

In [ ]:
def evaluate_classical_loso_one_subject(
        data,
        test_subject: str,
        model_name: str,
        seed: int,
        cfg,
):
    """
    Train a classical model on all subjects except the held-out one,
    then evaluate on the held-out subject.

    Important:
    For LOSO susceptibility, the held-out subject is the test subject.
    Classical features must follow cfg.feature, e.g. wide-timebin.
    """
    train_subjects = [s for s in data.subjects if s != test_subject]

    X_train, y_train, _ = concatenate_training_subjects_with_unique_groups(
        data,
        train_subjects,
    )

    X_test = data.X_by_sub[test_subject]
    y_test = data.y_by_sub[test_subject]

    F_train = make_classical_features(
        X_train,
        feature_mode=cfg.feature.mode,
        times=data.times,
        sfreq=data.sfreq,
        bin_ms=cfg.feature.bin_ms,
    )

    F_test = make_classical_features(
        X_test,
        feature_mode=cfg.feature.mode,
        times=data.times,
        sfreq=data.sfreq,
        bin_ms=cfg.feature.bin_ms,
    )

    clf = get_model(model_name)
    clf.fit(F_train, y_train)

    if hasattr(clf, "predict_proba"):
        scores = clf.predict_proba(F_test)[:, 1]
    else:
        scores = clf.decision_function(F_test)

    y_pred = clf.predict(F_test)

    crop_tmin, crop_tmax = cfg.effective_crop_window()

    row = {
        "subject": test_subject,
        "heldout_subject": test_subject,
        "seed": seed,
        "fold": 1,
        "model": model_name,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "train_pos_rate": float(y_train.mean()),
        "test_pos_rate": float(y_test.mean()),
        "best_epoch": np.nan,
        "best_val_auc": np.nan,
        "feature_mode": cfg.feature.mode,
        "bin_ms": int(cfg.feature.bin_ms) if cfg.feature.mode == "time_binned" else np.nan,
        "crop_tmin": float(crop_tmin),
        "crop_tmax": float(crop_tmax),
    }

    row.update(classification_metric_dict(y_test, scores, y_pred))

    return row

In [ ]:
def evaluate_eegnet_loso_one_subject(
        data,
        test_subject: str,
        seed: int,
        cfg,
        device,
):
    """
    Train EEGNet on all training subjects and test on one held-out subject.
    Validation is created only from the training subjects.
    """
    train_subjects = [s for s in data.subjects if s != test_subject]

    X_train, y_train, g_train = concatenate_training_subjects_with_unique_groups(data, train_subjects)
    X_test = data.X_by_sub[test_subject]
    y_test = data.y_by_sub[test_subject]

    result = train_eegnet_one_fold(
        X_train_fold=X_train,
        y_train_fold=y_train,
        groups_train_fold=g_train,
        X_test_fold=X_test,
        y_test_fold=y_test,
        device=device,
        normalize=cfg.eegnet.normalize,
        batch_size=cfg.eegnet.batch_size,
        lr=cfg.eegnet.lr,
        weight_decay=cfg.eegnet.weight_decay,
        epochs=cfg.eegnet.epochs,
        patience=cfg.eegnet.early_stopping_patience,
        val_size=cfg.splits.val_size,
        val_n_tries=cfg.splits.val_n_tries,
        use_class_weight=cfg.eegnet.use_class_weight,
        grad_clip_norm=cfg.eegnet.grad_clip_norm,
        eegnet_kwargs={
            "n_classes": cfg.eegnet.n_classes,
            "f1": cfg.eegnet.f1,
            "d": cfg.eegnet.d,
            "f2": cfg.eegnet.f2,
            "k1": cfg.eegnet.k1,
            "k2": cfg.eegnet.k2,
            "dropout": cfg.eegnet.dropout,
        },
        seed=seed,
        verbose=cfg.verbose,
    )

    test_metrics = result["test_metrics"]

    row = {
        "subject": test_subject,
        "seed": seed,
        "fold": 1,
        "model": "eegnet",
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "train_pos_rate": float(y_train.mean()),
        "test_pos_rate": float(y_test.mean()),
        "best_epoch": result["best_epoch"],
        "best_val_auc": result["best_val_auc"],
        "auc": float(test_metrics["auc"]),
        "pr_auc": float(test_metrics["pr_auc"]),
        "acc": float(test_metrics["acc"]),
        "bal_acc": float(test_metrics["bal_acc"]),
        "f1": float(test_metrics["f1"]),
        "tn": int(test_metrics["tn"]),
        "fp": int(test_metrics["fp"]),
        "fn": int(test_metrics["fn"]),
        "tp": int(test_metrics["tp"]),
    }

    crop_tmin, crop_tmax = cfg.effective_crop_window()

    row.update({
        "heldout_subject": test_subject,
        "feature_mode": "raw_epoch",
        "bin_ms": np.nan,
        "crop_tmin": float(crop_tmin),
        "crop_tmax": float(crop_tmax),
    })

    return row, result["history_df"]

In [ ]:
def run_loso_benchmark(data, cfg, device):
    """
    Run LOSO across all configured seeds and held-out subjects.

    Saves:
    - seed-level LOSO results under tables/folds/
    - EEGNet LOSO histories under histories/eegnet/
    """
    all_rows = []
    all_histories = {}

    for seed in cfg.seeds.seeds:
        print("#" * 90)
        print(f"LOSO SEED {seed}")
        print("#" * 90)

        seed_rows = []
        seed_histories = {}

        seed_fold_dir = cfg.table_dir("folds", seed=seed)
        seed_history_dir = cfg.history_dir("eegnet", seed=seed)

        for test_subject in data.subjects:
            print("=" * 80)
            print(f"HELD-OUT SUBJECT: {test_subject}")
            print("=" * 80)

            # Classical baselines
            row_lda = evaluate_classical_loso_one_subject(
                data=data,
                test_subject=test_subject,
                model_name="lda",
                seed=seed,
                cfg=cfg,
            )

            row_logreg = evaluate_classical_loso_one_subject(
                data=data,
                test_subject=test_subject,
                model_name="logreg",
                seed=seed,
                cfg=cfg,
            )

            # EEGNet
            row_eegnet, hist_df = evaluate_eegnet_loso_one_subject(
                data=data,
                test_subject=test_subject,
                seed=seed,
                cfg=cfg,
                device=device,
            )

            seed_rows.extend([row_lda, row_logreg, row_eegnet])
            all_rows.extend([row_lda, row_logreg, row_eegnet])

            seed_histories[test_subject] = hist_df
            seed_histories[test_subject].to_csv(
                seed_history_dir / f"eegnet_history_heldout-subject-{test_subject}.csv",
                index=False,
            )

        df_seed = pd.DataFrame(seed_rows)

        save_dataframe(
            df_seed,
            seed_fold_dir / "loso_fold_results.csv",
        )

        all_histories[seed] = seed_histories

    df_loso = pd.DataFrame(all_rows)
    return df_loso, all_histories

## 4. Run LOSO across seeds and held-out subjects

In [ ]:
df_loso_all, loso_histories_by_seed = run_loso_benchmark(
    data=data_loso,
    cfg=cfg_loso,
    device=device,
)

LOSO_SUMMARY_DIR = cfg_loso.table_dir("summaries")

save_dataframe(
    df_loso_all,
    LOSO_SUMMARY_DIR / "loso_all_results.csv",
)

print("LOSO result shape:", df_loso_all.shape)

display(
    round_numeric_columns(df_loso_all.head(), decimals=4)
)

## 5. Summarize LOSO results across seeds

In [ ]:
df_loso_subject_seed = summarize_fold_results(
    df_loso_all,
    group_cols=("subject", "model", "seed"),
)

save_dataframe(
    df_loso_subject_seed,
    LOSO_SUMMARY_DIR / "loso_subject_seed_summary.csv",
)

display(
    round_numeric_columns(df_loso_subject_seed, decimals=4)
)

In [ ]:
df_loso_subject_across_seeds = summarize_subject_results_across_seeds(
    df_loso_subject_seed
)

save_dataframe(
    df_loso_subject_across_seeds,
    LOSO_SUMMARY_DIR / "loso_subject_summary_across_seeds.csv",
)

display(
    round_numeric_columns(df_loso_subject_across_seeds, decimals=4)
)

In [ ]:
loso_subject_auc_export = df_loso_subject_across_seeds.copy()

dataset_label = (
    f"{cfg_loso.dataset.dataset}_{cfg_loso.dataset.event_preset}"
    if cfg_loso.dataset.dataset == "ds006480"
    else cfg_loso.dataset.dataset
)

loso_subject_auc_export["dataset"] = cfg_loso.dataset.dataset
loso_subject_auc_export["task"] = cfg_loso.dataset.task
loso_subject_auc_export["event_preset"] = cfg_loso.dataset.event_preset
loso_subject_auc_export["dataset_label"] = dataset_label
loso_subject_auc_export["heldout_subject"] = loso_subject_auc_export["subject"].astype(str)
loso_subject_auc_export["feature_mode"] = cfg_loso.feature.mode
loso_subject_auc_export["bin_ms"] = int(cfg_loso.feature.bin_ms)
loso_subject_auc_export["crop_tmin"] = cfg_loso.effective_crop_window()[0]
loso_subject_auc_export["crop_tmax"] = cfg_loso.effective_crop_window()[1]
loso_subject_auc_export["run_tag"] = cfg_loso.run_tag
loso_subject_auc_export["experiment_name"] = cfg_loso.name

loso_subject_auc_export = loso_subject_auc_export.rename(
    columns={
        "auc_seed_mean": "loso_auc",
        "auc_seed_std": "loso_auc_std",
    }
)

save_dataframe(
    loso_subject_auc_export,
    LOSO_SUMMARY_DIR / "loso_subject_auc.csv",
)

display(round_numeric_columns(loso_subject_auc_export.head(), 4))

In [ ]:
df_loso_model_summary = summarize_model_results_across_subjects(
    df_loso_subject_across_seeds
)

save_dataframe(
    df_loso_model_summary,
    LOSO_SUMMARY_DIR / "loso_model_summary_across_subjects.csv",
)

display(
    round_numeric_columns(df_loso_model_summary, decimals=4)
)

## 6. Subject-wise LOSO ROC-AUC table

In [ ]:
table_loso_subject_auc = make_subject_metric_table(
    df_loso_subject_across_seeds.rename(columns={"auc_seed_mean": "auc_mean"}),
    metric_col="auc_mean",
)

display(table_loso_subject_auc)

save_dataframe(
    table_loso_subject_auc,
    cfg_loso.table_dir("reporting") / "loso_subject_auc_table.csv",
)

## 7. LOSO model ranking table

In [ ]:
table_loso_model_ranking = make_model_ranking_table(
    df_loso_model_summary,
    metric_col="mean_subject_auc",
)

display(table_loso_model_ranking)

save_dataframe(
    table_loso_model_ranking,
    cfg_loso.table_dir("reporting") / "loso_model_ranking_table.csv",
)

In [ ]:
print_subject_model_summary(
    df_loso_subject_across_seeds.rename(
        columns={
            "auc_seed_mean": "auc_mean",
            "auc_seed_std": "auc_std",
        }
    ),
    metric_mean_col="auc_mean",
    metric_std_col="auc_std",
)

In [ ]:
print_model_ranking(
    df_loso_model_summary,
    metric_col="mean_subject_auc",
)

## 8. LOSO ROC-AUC plot by held-out subject

In [ ]:
fig_loso_auc = plot_subject_metric_bars(
    df_loso_subject_across_seeds.rename(
        columns={
            "auc_seed_mean": "auc_mean",
            "auc_seed_std": "auc_std",
        }
    ),
    metric_col="auc_mean",
    error_col="auc_std",
    title=f"LOSO benchmark: ROC-AUC by held-out subject — {cfg_loso.dataset.dataset}, {cfg_loso.dataset.task}",
    ylabel="ROC-AUC",
)
fig_loso_auc

## 9. LOSO seed stability plot

In [ ]:
fig_loso_seed = plot_subject_seed_stability(
    df_loso_subject_across_seeds,
    metric_col="auc_seed_mean",
    error_col="auc_seed_std",
    title=f"LOSO benchmark: seed stability — {cfg_loso.dataset.dataset}, {cfg_loso.dataset.task}",
    ylabel="ROC-AUC across seeds",
)
fig_loso_seed

## 10. LOSO model ranking plot

In [ ]:
fig_loso_rank = plot_model_ranking_bars(
    df_loso_model_summary,
    metric_col="mean_subject_auc",
    title=f"LOSO model ranking by mean held-out-subject ROC-AUC — {cfg_loso.dataset.dataset}, {cfg_loso.dataset.task}",
    ylabel="Mean held-out-subject ROC-AUC",
)
fig_loso_rank

## 11. Compare within-subject benchmark vs LOSO

This comparison highlights how much performance drops when the test subject is completely unseen.

In [ ]:
MAIN_SUMMARY_DIR = cfg_main.table_dir("summaries")

df_main_subject_across_seeds = pd.read_csv(
    MAIN_SUMMARY_DIR / "subject_summary_across_seeds.csv"
)

df_main_model_summary = pd.read_csv(
    MAIN_SUMMARY_DIR / "model_summary_across_subjects.csv"
)

display(
    round_numeric_columns(df_main_model_summary, decimals=4)
)

In [ ]:
df_main_vs_loso = make_main_vs_control_table(
    df_main_subject_across_seeds,
    df_loso_subject_across_seeds,
    value_col_main="auc_seed_mean",
    value_col_control="auc_seed_mean",
    main_label="within_subject",
    control_label="loso",
)

LOSO_COMPARISON_DIR = cfg_loso.table_dir("comparisons")

df_main_vs_loso = round_numeric_columns(df_main_vs_loso, decimals=4)

display(df_main_vs_loso)

save_dataframe(
    df_main_vs_loso,
    LOSO_COMPARISON_DIR / "within_subject_vs_loso_subject_table.csv",
)

## 12. Plot within-subject benchmark vs LOSO

In [ ]:
fig_main_vs_loso = plot_main_vs_control(
    df_main_vs_loso,
    main_col="within_subject_value",
    control_col="loso_value",
    title=f"Within-subject benchmark vs LOSO — {cfg_loso.dataset.dataset}, {cfg_loso.dataset.task}",
)
fig_main_vs_loso

## 13. Inspect one representative EEGNet LOSO learning curve

In [ ]:
example_seed = cfg_loso.seeds.seeds[0]
example_subject = data_loso.subjects[0]

hist_example_loso = loso_histories_by_seed[example_seed][example_subject]
hist_example_loso.head()

In [ ]:
fig_hist_loso = plot_learning_curve(
    hist_example_loso,
    title=f"EEGNet LOSO learning curve — seed {example_seed}, held-out subject {example_subject}",
)
fig_hist_loso

## 14. Save key LOSO figures

In [ ]:
LOSO_BENCHMARK_FIG_DIR = cfg_loso.figure_dir("benchmark")
LOSO_LEARNING_CURVE_FIG_DIR = cfg_loso.figure_dir("learning_curves")

save_figure(
    fig_loso_auc,
    LOSO_BENCHMARK_FIG_DIR / "loso_auc_by_subject.png",
)

save_figure(
    fig_loso_seed,
    LOSO_BENCHMARK_FIG_DIR / "loso_seed_stability.png",
)

save_figure(
    fig_loso_rank,
    LOSO_BENCHMARK_FIG_DIR / "loso_model_ranking.png",
)

save_figure(
    fig_main_vs_loso,
    LOSO_BENCHMARK_FIG_DIR / "within_subject_vs_loso.png",
)

save_figure(
    fig_hist_loso,
    LOSO_LEARNING_CURVE_FIG_DIR / "example_loso_learning_curve.png",
)

save_experiment_metadata(
    cfg_loso,
    extra={
        "notebook": "04_cross_subject_loso.ipynb",
        "status": "completed",
        "main_experiment_key": EXPERIMENT_NAME,
        "n_subjects": int(len(data_loso.subjects)),
        "n_seeds": int(len(cfg_loso.seeds.seeds)),
        "models": sorted(df_loso_all["model"].unique().tolist()),
        "n_loso_rows": int(len(df_loso_all)),
        "summary_dir": str(LOSO_SUMMARY_DIR),
        "comparison_dir": str(LOSO_COMPARISON_DIR),
        "benchmark_figure_dir": str(LOSO_BENCHMARK_FIG_DIR),
        "learning_curve_figure_dir": str(LOSO_LEARNING_CURVE_FIG_DIR),
    },
)

print("Saved LOSO summary tables to:", LOSO_SUMMARY_DIR)
print("Saved LOSO comparison tables to:", LOSO_COMPARISON_DIR)
print("Saved LOSO benchmark figures to:", LOSO_BENCHMARK_FIG_DIR)
print("Saved LOSO learning-curve figures to:", LOSO_LEARNING_CURVE_FIG_DIR)

## 15. LOSO interpretation checklist

When reviewing LOSO results, the expected pattern is:

- performance should generally be lower than the within-subject benchmark
- the drop quantifies how subject-specific the task is
- a stronger model should ideally:
  - remain competitive under LOSO
  - avoid catastrophic collapse on weak held-out subjects
  - show reasonable seed stability

LOSO is supporting evidence, not the sole basis for final model choice.

In [ ]:
print("=" * 80)
print("Within-subject model summary")
print("=" * 80)
display(round_numeric_columns(df_main_model_summary, decimals=4))

print("=" * 80)
print("LOSO model summary")
print("=" * 80)
display(round_numeric_columns(df_loso_model_summary, decimals=4))